## Topic: Practical Implementation of Vector Database with LangChain

In [ ]:
"""  - Diagram or Process or Architecture


- 1. Raw Data -> extract docs 
                        -> 2. chunk docs -> ....
                                  ↓
                        -> 3. embedding chunk -> ....
                                  ↓ 
                        -> 4. vector embedding -> ....
                                  ↓ 
                        - 5.  Vector Database -> ....




# 1. Raw Data (docs): Text file, pdf file, docs file, images
    - extract docs: load the entire raw data 

# 2. chunking docs: 
    - split of the docs into smaller part.
    - solve the input limitation(content window)
    - model can better memories the docs 

# 3. embedding chunk
    - every larger language model has embedding model
    

# 4. vector embedding
    - convert chunk of embedding(vector or numerical number) of chunk of docs


# 5. Vector Database or Knowledge Base
    - Store the embedded vector into vector database
    - this is also know as Knowledge Base (RAG Base system)

    - we use various type of vector database
        - 1. Chroma (locally store the data)
        
        - 2. FAISS (locally store the data)
        
        - 3. Pinecone (cloud store the data)

"""

In [ ]:
""" 
        Documents
            ↓
        Document Loader
            ↓
        Text Splitting
            ↓
        Embedding Model
            ↓
        Vector Embeddings
            ↓
        Vector Database

"""

###  1. ChromaDB (locally store the data)

In [ ]:
"""  
            - Complete Workflow Diagram
            ================================

┌─────────────────────────────────────────────────────────────────────────────┐
│                    COMPLETE CHROMADB WORKFLOW DIAGRAM                       │
│                                                                             │
│  ╔═══════════════════════════════════════════════════════════════════════╗  │
│  ║                    PHASE 1: SETUP & INITIALIZATION                    ║  │
│  ╚═══════════════════════════════════════════════════════════════════════╝  │
│                                                                             │
│  ┌──────────────┐     ┌──────────────────┐     ┌───────────────────┐        │
│  │ Install      │────→│ Initialize       │────→│ Create/Get        │        │
│  │ ChromaDB     │     │ Client           │     │ Collection        │        │
│  │              │     │                  │     │                   │        │
│  │ pip install  │     │ • In-memory      │     │ • Name            │        │
│  │ chromadb     │     │ • Persistent     │     │ • Embedding Func  │        │
│  │              │     │ • Client-Server  │     │ • Distance Metric │        │
│  └──────────────┘     └──────────────────┘     └────────┬──────────┘        │
│                                                          │                  │
│  ╔═══════════════════════════════════════════════════════╧═══════════════╗  │
│  ║                    PHASE 2: DATA INGESTION                            ║  │
│  ╚═══════════════════════════════════════════════════════════════════════╝  │
│                                                                             │
│  ┌──────────────┐     ┌──────────────────┐     ┌───────────────────┐        │
│  │ Raw Data     │────→│ Preprocessing    │────→│ Chunking          │        │
│  │              │     │                  │     │                   │        │
│  │ • Documents  │     │ • Clean text     │     │ • Fixed size      │        │
│  │ • PDFs       │     │ • Remove noise   │     │ • Sentence-based  │        │
│  │ • Web pages  │     │ • Normalize      │     │ • Semantic chunks │        │
│  │ • CSVs       │     │ • Extract text   │     │ • Overlapping     │        │
│  └──────────────┘     └──────────────────┘     └────────┬──────────┘        │
│                                                          │                  │
│                                                          ▼                  │
│  ┌───────────────────────────────────────────────────────────────────┐      │
│  │                    EMBEDDING GENERATION                           │      │
│  │                                                                   │      │
│  │   "The cat sat on the mat"                                        │      │
│  │          │                                                        │      │
│  │          ▼                                                        │      │
│  │   ┌──────────────────┐                                            │      │
│  │   │ Embedding Model  │  ChromaDB Default: all-MiniLM-L6-v2        │      │
│  │   │                  │  OR: OpenAI, Cohere, HuggingFace, etc.     │      │
│  │   └────────┬─────────┘                                            │      │
│  │            ▼                                                      │      │
│  │   [0.023, -0.041, 0.087, ..., 0.012]  (384 dimensions)            │      │
│  └───────────────────────────────────────────────────────────────────┘      │
│                          │                                                  │
│                          ▼                                                  │
│  ┌───────────────────────────────────────────────────────────────────┐      │
│  │                    STORAGE IN COLLECTION                          │      │
│  │                                                                   │      │
│  │   collection.add(                                                 │      │
│  │       ids        = ["doc1", "doc2", ...]     ← Unique IDs         │      │
│  │       documents  = ["text1", "text2", ...]   ← Raw text           │      │
│  │       embeddings = [[0.02, ...], ...]        ← Optional vectors   │      │
│  │       metadatas  = [{"src": "wiki"}, ...]    ← Metadata           │      │
│  │   )                                                               │      │
│  │                                                                   │      │
│  │   ┌─────────┐  ┌──────────────┐  ┌───────────────┐                │      │
│  │   │ Vector  │  │  Metadata    │  │  Document     │                │      │
│  │   │ Index   │  │  Store       │  │  Store        │                │      │
│  │   │ (HNSW)  │  │  (SQLite)   │  │  (SQLite)    │                  │      │
│  │   └─────────┘  └──────────────┘  └───────────────┘                │      │
│  └───────────────────────────────────────────────────────────────────┘      │
│                                                                             │
│  ╔═══════════════════════════════════════════════════════════════════════╗  │
│  ║                    PHASE 3: QUERYING                                  ║  │
│  ╚═══════════════════════════════════════════════════════════════════════╝  │
│                                                                             │
│  ┌──────────────┐     ┌──────────────────┐     ┌───────────────────┐        │
│  │ User Query   │────→│ Embed Query      │────→│ ANN Search        │        │
│  │              │     │ (same model)     │     │ (HNSW traversal)  │        │
│  │ "Tell me     │     │                  │     │                   │        │
│  │  about cats" │     │ [0.03, -0.05,..] │     │ Find nearest K    │        │
│  └──────────────┘     └──────────────────┘     │ vectors           │        │
│                                                └────────┬──────────┘        │
│                                                         │                   │
│                       ┌──────────────────┐              │                   │
│                       │ Apply Filters    │◄─────────────┘                   │
│                       │ (Optional)       │                                  │
│                       │                  │                                  │
│                       │ where={"src":    │                                  │
│                       │   "wikipedia"}   │                                  │
│                       └────────┬─────────┘                                  │
│                                │                                            │
│                                ▼                                            │
│                       ┌──────────────────┐                                  │
│                       │ Return Results   │                                  │
│                       │                  │                                  │
│                       │ • Documents      │                                  │
│                       │ • Distances      │                                  │
│                       │ • Metadatas      │                                  │
│                       │ • IDs            │                                  │
│                       └──────────────────┘                                  │
│                                                                             │
│  ╔═══════════════════════════════════════════════════════════════════════╗  │
│  ║                    PHASE 4: APPLICATION INTEGRATION                   ║  │
│  ╚═══════════════════════════════════════════════════════════════════════╝  │
│                                                                             │
│  ┌──────────────────────────────────────────────────────────────────┐       │
│  │                                                                  │       │
│  │   Retrieved Context + User Query ──→ LLM (GPT/Claude) ──→ Answer │       │
│  │                                                                  │       │
│  │   OR: Direct similarity results for search/recommendation        │       │
│  │                                                                  │       │
│  └──────────────────────────────────────────────────────────────────┘       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

"""

In [ ]:
"""    ChromaDB Architecture Deep Dive 

┌─────────────────────────────────────────────────────────── ┐
│                   CHROMADB INTERNALS                       │
│                                                            │
│  ┌──────────────────────────────────────────────────────┐  │
│  │                    Client Layer                      │  │
│  │                                                      │  │
│  │   ┌──────────────┐  ┌─────────────┐  ┌───────────┐   │  │
│  │   │ EphemeralClient││PersistentClient│ │HttpClient   │  │  
│  │   │ (in-memory)  │  │(local disk)  │ │(server)   │   │  │
│  │   └──────────────┘  └─────────────┘  └───────────┘   │  │
│  └─────────────────────────┬────────────────────────────┘  │
│                             │                              │
│  ┌─────────────────────────▼────────────────────────────┐  │
│  │                  Collection Manager                  │  │
│  │                                                      │  │
│  │   Collection = {                                     │  │
│  │       name: "my_collection",                         │  │
│  │       embedding_function: SentenceTransformer,       │  │
│  │       metadata: {"hnsw:space": "cosine"}             │  │
│  │   }                                                  │  │
│  └─────────────────────────┬────────────────────────────┘  │
│                             │                              │
│  ┌─────────────────────────▼────────────────────────────┐  │
│  │                   Storage Layer                      │  │
│  │                                                      │  │
│  │   ┌─────────────────┐    ┌──────────────────────┐    │  │
│  │   │  HNSW Index     │    │  SQLite Database     │    │  │
│  │   │  (hnswlib)      │    │                      │    │  │
│  │   │                 │    │  ┌────────────────┐  │    │  │
│  │   │  • Vector data  │    │  │ Documents      │  │    │  │
│  │   │  • Graph struct │    │  │ table          │  │    │  │
│  │   │  • ANN search   │    │  ├────────────────┤  │    │  │
│  │   │                 │    │  │ Metadata       │  │    │  │
│  │   │  Parameters:    │    │  │ table          │  │    │  │
│  │   │  • M = 16       │    │  ├────────────────┤  │    │  │
│  │   │  • ef_const=100 │    │  │ Embeddings     │  │    │  │
│  │   │  • ef_search=10 │    │  │ reference      │  │    │  │
│  │   │                 │    │  └────────────────┘  │    │  │
│  │   └─────────────────┘    └──────────────────────┘    │  │
│  └──────────────────────────────────────────────────────┘  │
│                                                            │
│  ┌──────────────────────────────────────────────────────┐  │
│  │              Embedding Functions                     │  │
│  │                                                      │  │
│  │  Default: all-MiniLM-L6-v2 (Sentence Transformers)   │  │
│  │                                                      │  │
│  │  Built-in:                                           │  │
│  │  ├── OpenAIEmbeddingFunction                         │  │
│  │  ├── CohereEmbeddingFunction                         │  │
│  │  ├── HuggingFaceEmbeddingFunction                    │  │
│  │  ├── GoogleGenerativeAiEmbeddingFunction             │  │
│  │  ├── InstructorEmbeddingFunction                     │  │
│  │  └── Custom (any callable)                           │  │
│  └──────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────── ┘


"""